# PCIbex experiment results

Run the cells in order. The notebook reads available `results_01.csv` through `results_12.csv` (also accepting `result_XX.csv`) from the directory containing `stimuli.csv` and combines them. Missing lists are reported; duplicate filenames for the same list are an error. `list_number` identifies the results file, `file_session` is the submission number within that file, and `session` is a unique submission number across the currently available files. Session numbers can change when earlier list files are added; use `list_number` and `file_session` to identify a submission within its source file. They distinguish separate submissions, even if a participant returns.

After combining results, the notebook reports Prolific IDs appearing in more than one list and submissions with no usable Prolific ID. This check covers the available files and identifies appearances, not necessarily completed lists.

`event_table` contains one row per **logged** experimental keystroke, with trial metadata and choices alongside each event. Practice and instructions are excluded.

**Timing:** all times are milliseconds and are calculated in chronological order before sorting the table. `since_previous_key_ms` is the elapsed time since the preceding logged experimental key in the same submission, including across trials. `within_trial_ms` resets at each trial. These columns include context and question events. The context key is the first logged key in its trial, so its within-trial interval is missing; the cross-trial interval is not a context reading time. Question intervals can include transitions or fixed delays. Region reading times are the interval from the preceding region key to that region's key: for example, `r1_ms` runs from `r0` to `r1`. `r0` starts the sentence, not a word. The interval before the choice includes the 500 ms fixation and is **not** a choice-onset RT. As in the script, `cb` equal to `y` swaps the sides; every other value, including a blank, uses A on the left.

The preview space key has no `.log()` in the script. It, other unlogged keys, exact display-onset times and consequently true preview/choice-onset RTs cannot be reconstructed. This notebook does not infer or fabricate them. The file contains participant identifiers; keep any exports private.

## 1. Setup

Load the result files and configure the print-friendly plot style. Run the notebook from this directory or the repository root.

In [63]:
import csv
import re
from pathlib import Path
import pandas as pd
from IPython.display import display

# Works when the notebook is run from either new/ or the repository root.
RESULTS_DIR = Path('.') if Path('stimuli.csv').is_file() else Path('new')
if not (RESULTS_DIR / 'stimuli.csv').is_file():
    raise FileNotFoundError(f'No stimuli.csv found in {RESULTS_DIR}')

# Read the PCIbex export files, which have numbered column descriptions that change from file to file. The first block of lines starting with "# Results on" is session 1, the next block is session 2, etc. Each block has its own column descriptions, which are numbered and may be in a different order than other blocks. The export files have a UTF-8 BOM, so we use utf-8-sig to read them.
def read_pcibex(path):
    """Read the numbered, changing column descriptions in a PCIbex export."""
    records = []
    columns = {}
    session = 0
    with path.open(encoding='utf-8-sig', newline='') as source:
        for line_number, line in enumerate(source, 1):
            if line.startswith('# Results on '):
                session += 1
                columns = {}
            match = re.match(r'^#\s*(\d+)\.\s*(.*?)\.\s*$', line)
            if match:
                columns[int(match.group(1)) - 1] = match.group(2)
            elif line.startswith('#') or not line.strip():
                continue
            else:
                if not session:
                    raise ValueError(f'No results block before line {line_number}')
                values = next(csv.reader([line]))
                names = [columns.get(i) for i in range(len(values))]
                if any(name is None for name in names):
                    raise ValueError(f'Missing column description at line {line_number}')
                # participant_id appears twice: typed ID first, URL parameter second.
                seen = {}
                unique_names = []
                for name in names:
                    seen[name] = seen.get(name, 0) + 1
                    unique_names.append(name if seen[name] == 1 else f'{name}_{seen[name]}')
                record = dict(zip(unique_names, values))
                record.update(session=session, source_line=line_number)
                records.append(record)
    if not records:
        raise ValueError(f'No result rows in {path}')
    return pd.DataFrame(records)

# Combine the available lists while keeping file-specific submission identifiers.
result_frames = []
missing_lists = []
session_offset = 0
for list_number in range(1, 13):
    candidates = [RESULTS_DIR / f'{prefix}_{list_number:02d}.csv' for prefix in ('results', 'result')]
    paths = [path for path in candidates if path.is_file()]
    if len(paths) > 1:
        raise ValueError(f'Multiple result files for list {list_number:02d}: {paths}')
    if not paths:
        missing_lists.append(list_number)
        continue
    path = paths[0]
    result = read_pcibex(path)
    result['file_session'] = result['session']
    result['session'] += session_offset
    result['list_number'] = list_number
    result['source_file'] = path.name
    session_offset = int(result['session'].max())
    result_frames.append(result)
    print(f'List {list_number:02d}: {path.name}, {len(result):,} rows, {result.session.nunique()} submissions')

# Combine the results into a single DataFrame. The session numbers are now unique across files, and the list number is recorded for each row.
if not result_frames:
    raise FileNotFoundError(f'No result_XX.csv or results_XX.csv files for lists 01-12 in {RESULTS_DIR}')
raw = pd.concat(result_frames, ignore_index=True)
print('Missing lists:', ', '.join(f'{number:02d}' for number in missing_lists) if missing_lists else 'None')
print(f'Combined: {len(raw):,} rows in {raw.session.nunique()} submission blocks')

List 01: results_01.csv, 5,800 rows, 10 submissions
List 02: results_02.csv, 5,730 rows, 10 submissions
List 03: results_03.csv, 5,730 rows, 10 submissions
List 04: results_04.csv, 5,730 rows, 10 submissions
Missing lists: 05, 06, 07, 08, 09, 10, 11, 12
Combined: 22,990 rows in 40 submission blocks


In [64]:
import math
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

# 700 px at 96 dpi is about 185 mm: within the width of an A4 page.
FIGURE_WIDTH = 700
FIGURE_HEIGHT = 440
FONT_FAMILY = 'Times New Roman, Times, serif'
CONDITION_COLORS = {
    'ctx-exhaustive': '#5e81ac', 'pvf-exhaustive': '#5e81ac',
    'ctx-base': '#a3be8c', 'pvf-base': '#a3be8c',
    'ctx-contrastive': '#ebcb8b', 'pvf-contrastive': '#ebcb8b',
    'non-focused': '#4c566a',
}
TARGET_DASHES = {
    'non-focused': 'solid', 'pvf-exhaustive': 'dash',
    'pvf-base': 'dot', 'pvf-contrastive': 'dashdot',
}
CHOICE_COLORS = ['#88c0d0', '#d08770']  # A is purple; B is orange.

nord_template = go.layout.Template()
nord_template.layout.paper_bgcolor = 'white'
nord_template.layout.plot_bgcolor = 'white'
nord_template.layout.colorway = ['#5e81ac', '#a3be8c', '#ebcb8b', '#4c566a']
nord_template.layout.font = dict(family=FONT_FAMILY, size=13, color='black')
nord_template.layout.title = dict(font=dict(size=16, color='black'))
nord_template.layout.legend = dict(font=dict(size=11), bgcolor='rgba(255,255,255,0.8)')
nord_template.layout.hoverlabel = dict(font=dict(family=FONT_FAMILY, size=12, color='black'))
axis_defaults = dict(
    title=dict(font=dict(size=13, color='black')),
    tickfont=dict(size=11, color='black'),
    gridcolor='rgba(0,0,0,0.08)', zerolinecolor='rgba(0,0,0,0.12)',
    linecolor='rgba(0,0,0,0.4)', automargin=True
)
nord_template.layout.xaxis = axis_defaults
nord_template.layout.yaxis = axis_defaults
nord_template.layout.annotationdefaults = dict(font=dict(family=FONT_FAMILY, size=11, color='black'))
nord_template.data.scatter = [go.Scatter(line=dict(width=1.5), marker=dict(size=6))]
pio.templates['nord_light_paper'] = nord_template
pio.templates.default = 'nord_light_paper'

## 2. Preprocessing

The following checks report issues but do not automatically remove submissions or failed attention checks.

### 2.1. Checking for repeating participants

The cross-list check covers available files and reports IDs appearing in more than one list, not necessarily completed lists or repeats within one list. Missing IDs cannot be checked.

In [65]:
# Identify IDs present in more than one available list; do not exclude them.
if 'prolific_pid' not in raw.columns:
    raise ValueError('The combined results do not contain a prolific_pid column; cross-list participation cannot be checked')

participant_lists = raw[['session', 'list_number', 'source_file', 'file_session', 'prolific_pid']].copy()
participant_lists['prolific_pid'] = participant_lists['prolific_pid'].astype('string').str.strip()
participant_lists.loc[
    participant_lists['prolific_pid'].str.lower().isin(['', 'null', 'undefined', 'nan']), 'prolific_pid'
] = pd.NA
participant_lists = participant_lists.dropna(subset=['prolific_pid']).drop_duplicates()

list_counts = participant_lists.groupby('prolific_pid')['list_number'].nunique()
multi_list_ids = list_counts.index[list_counts.gt(1)]
multi_list_participants = participant_lists.loc[
    participant_lists['prolific_pid'].isin(multi_list_ids)
].sort_values(['prolific_pid', 'list_number', 'file_session'])

print(f'Participants appearing in more than one list: {len(multi_list_ids)}')
if not multi_list_participants.empty:
    with pd.option_context('display.max_rows', None, 'display.max_columns', None):
        display(multi_list_participants)

missing_pid_sessions = raw['session'].nunique() - participant_lists['session'].nunique()
print(f'Submissions without a usable prolific_pid: {missing_pid_sessions}; these cannot be checked for cross-list participation')

Participants appearing in more than one list: 0
Submissions without a usable prolific_pid: 0; these cannot be checked for cross-list participation


### 2.2. Checking attention checks

First reconstruct trial and event records from the raw log; then compare each attention-check choice with the recorded correct answer. Missing responses count as failures. Flagged submissions remain in the analysis.

In [66]:
# Reconstruct experimental trials and logged key events before checking attention.
trial_keys = ['session', 'Order number of item', 'Inner element number']
metadata = ['list_number', 'source_file', 'file_session', 'participant_id', 'prolific_pid', 'no', 'item', 'exp', 'cnd',
            'counterbalancing', 'a', 'b', 'correct', 'vpx', 'def', 'pst', 'mlt', 'alt']
experiment = raw.loc[raw['Label'].eq('experiment')].copy()
if experiment.empty:
    raise ValueError('No experiment rows found')
if 'counterbalancing' not in experiment:
    raise ValueError('Counterbalancing column is missing from the results')
experiment['EventTime'] = pd.to_numeric(experiment['EventTime'], errors='raise')
for column in metadata:
    if column not in experiment:
        experiment[column] = pd.NA
    experiment[column] = experiment[column].replace({'NULL': pd.NA, '': pd.NA})

# Order number identifies the design item, not its randomized presentation order.
starts = experiment.loc[experiment['Parameter'].eq('_Trial_') & experiment['Value'].eq('Start')].copy()
if starts.duplicated(trial_keys).any():
    raise ValueError('Repeated trial identifiers in a submission; inspect the raw results')
starts = starts.sort_values(['session', 'EventTime', 'source_line'])
starts['trial_number'] = starts.groupby('session').cumcount() + 1
trials = starts[trial_keys + ['trial_number', 'EventTime'] + metadata].rename(
    columns={'EventTime': 'trial_start_ms', 'counterbalancing': 'cb', 'a': 'option_a', 'b': 'option_b'})
if trials.empty:
    raise ValueError('No experimental trial starts found')
for column in ['no', 'item']:
    trials[column] = pd.to_numeric(trials[column], errors='raise').astype('Int64')
for column in ['exp', 'pst', 'mlt']:
    trials[column] = pd.to_numeric(trials[column], errors='raise').astype('Int64')

# Extract the logged key events, which are used to calculate reading times and choice intervals.
events = experiment.loc[experiment['PennElementType'].eq('Key') & experiment['Parameter'].eq('PressedKey'),
                        trial_keys + ['source_line', 'EventTime', 'PennElementName', 'Value']].copy()
events = events.rename(columns={'EventTime': 'timestamp_ms', 'PennElementName': 'stage', 'Value': 'key'})
events['key'] = events['key'].map(lambda x: 'SPACE' if x == ' ' else x.upper())
events = events.merge(trials, on=trial_keys, how='left', validate='many_to_one', indicator=True)
if not events['_merge'].eq('both').all():
    raise ValueError('Some key events have no matching experimental trial start')
events = events.drop(columns='_merge').sort_values(['session', 'timestamp_ms', 'source_line']).reset_index(drop=True)
events['key_number_in_trial'] = events.groupby(trial_keys, sort=False).cumcount() + 1
events['since_previous_key_ms'] = events.groupby('session')['timestamp_ms'].diff()
events['within_trial_ms'] = events.groupby(trial_keys, sort=False)['timestamp_ms'].diff()
events['previous_stage'] = events.groupby(trial_keys, sort=False)['stage'].shift()
if (events['timestamp_ms'] < events['trial_start_ms']).any():
    raise ValueError('A key event precedes its trial start')

# Interpret the logged choice exactly as in main.js: y swaps left and right.
choices = events.loc[events.stage.eq('choice_key')].copy()
if choices.duplicated(trial_keys).any():
    raise ValueError('More than one logged choice in a trial; inspect events')
if not choices['key'].isin(['F', 'J']).all():
    raise ValueError('Unexpected choice key; inspect events')
choices['choice_side'] = choices['key'].map({'F': 'left', 'J': 'right'})
swapped = choices['cb'].eq('y').fillna(False)
is_a = (choices['key'].eq('J') & swapped) | (choices['key'].eq('F') & ~swapped)
choices['choice_ab'] = pd.Series('B', index=choices.index)
choices.loc[is_a, 'choice_ab'] = 'A'
choices['chosen_image'] = choices['option_a'].where(is_a, choices['option_b'])
choice_columns = trial_keys + ['key', 'choice_side', 'choice_ab', 'chosen_image', 'within_trial_ms']
trials = trials.merge(choices[choice_columns].rename(columns={'key': 'choice_key', 'within_trial_ms': 'interval_before_choice_ms'}),
                      on=trial_keys, how='left', validate='one_to_one')

# Only calculate a region RT when the preceding region key was actually logged.
for region in range(1, 7):
    current, previous = f'r{region}', f'r{region - 1}'
    region_events = events.loc[events.stage.eq(current) & events.previous_stage.eq(previous), trial_keys + ['within_trial_ms']]
    if region_events.duplicated(trial_keys).any():
        raise ValueError(f'Multiple {current} reading times in a trial')
    trials = trials.merge(region_events.rename(columns={'within_trial_ms': f'{current}_ms'}),
                          on=trial_keys, how='left', validate='one_to_one')

trials = trials.sort_values(['session', 'trial_number']).reset_index(drop=True)
print(f'{trials.session.nunique()} submissions; {len(trials)} trials; {len(events)} logged keys')
print(f"Trials without a logged choice: {trials.choice_key.isna().sum()}; trials with blank cb (treated as non-y): {trials.cb.isna().sum()}")
print('Logged stages:', events.stage.value_counts().to_dict())

40 submissions; 1600 trials; 14320 logged keys
Trials without a logged choice: 0; trials with blank cb (treated as non-y): 640
Logged stages: {'r0': 1600, 'r1': 1600, 'r5': 1600, 'r2': 1600, 'r3': 1600, 'r4': 1600, 'choice_key': 1600, 'r6': 1600, 'context_key': 960, 'question_key': 560}


In [67]:
# Attention checks are identified by their condition label, not by the optional event-table filter.
attention_checks = trials.loc[
    trials['cnd'].astype('string').str.startswith('attention', na=False)
].copy()
if attention_checks.empty:
    raise ValueError('No attention-check trials found')

correct_a = attention_checks['correct'].eq(attention_checks['option_a'])
correct_b = attention_checks['correct'].eq(attention_checks['option_b'])
if not (correct_a ^ correct_b).all():
    raise ValueError('Each attention-check correct answer must match exactly one of option_a or option_b')

attention_checks['correct_choice_ab'] = correct_a.map({True: 'A', False: 'B'})
attention_checks['correct_answer'] = attention_checks['correct']
swapped = attention_checks['cb'].eq('y').fillna(False)
correct_on_right = (attention_checks['correct_choice_ab'].eq('A') & swapped) | (
    attention_checks['correct_choice_ab'].eq('B') & ~swapped
)
attention_checks['correct_key'] = correct_on_right.map({True: 'J', False: 'F'})
attention_checks['passed'] = attention_checks['chosen_image'].eq(attention_checks['correct_answer']).fillna(False)
attention_checks['result'] = attention_checks['passed'].map({True: 'passed', False: 'FAILED / no response'})

attention_summary = (
    attention_checks.groupby(['session', 'list_number', 'source_file', 'file_session', 'participant_id', 'prolific_pid'], dropna=False)
    .agg(n_attention_checks=('item', 'size'), n_failed=('passed', lambda answers: int((~answers).sum())))
    .reset_index()
)
flagged_participants = attention_summary.loc[attention_summary['n_failed'].gt(0)].copy()
attention_responses = attention_checks[[
    'session', 'list_number', 'source_file', 'file_session', 'participant_id', 'prolific_pid', 'item', 'cnd', 'choice_key',
    'chosen_image', 'correct_answer', 'result'
]].sort_values(['session', 'item'], kind='stable')
failed_checks = attention_checks.loc[~attention_checks['passed'], [
    'session', 'list_number', 'source_file', 'file_session', 'participant_id', 'prolific_pid', 'item', 'cnd', 'choice_key',
    'chosen_image', 'correct_choice_ab', 'correct_answer', 'correct_key', 'result'
]].sort_values(['session', 'item'], kind='stable')

print('Attention-check responses by participant (chosen answer and result):')
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(attention_responses)

print('Participants with one or more failed attention checks:')
if flagged_participants.empty:
    print('None')
else:
    display(flagged_participants)
    print('Failed attention-check responses and correct answers:')
    with pd.option_context('display.max_rows', None, 'display.max_columns', None):
        display(failed_checks)


Attention-check responses by participant (chosen answer and result):


,session,list_number,source_file,file_session,participant_id,prolific_pid,item,cnd,choice_key,chosen_image,correct_answer,result
15,1,1,results_01.csv,1,69f05ebc11ace319136e99d2,69f05ebc11ace319136e99d2,1,attention,F,burgonya,burgonya,passed
35,1,1,results_01.csv,1,69f05ebc11ace319136e99d2,69f05ebc11ace319136e99d2,2,attention-alt,F,jegesmedve,jegesmedve,passed
39,1,1,results_01.csv,1,69f05ebc11ace319136e99d2,69f05ebc11ace319136e99d2,3,attention,F,pingvin,pingvin,passed
9,1,1,results_01.csv,1,69f05ebc11ace319136e99d2,69f05ebc11ace319136e99d2,4,attention-alt,F,vacsora,vacsora,passed
77,2,1,results_01.csv,2,6106fc335b80e1efc80ede21,6106fc335b80e1efc80ede21,1,attention,F,burgonya,burgonya,passed
76,2,1,results_01.csv,2,6106fc335b80e1efc80ede21,6106fc335b80e1efc80ede21,2,attention-alt,F,jegesmedve,jegesmedve,passed
58,2,1,results_01.csv,2,6106fc335b80e1efc80ede21,6106fc335b80e1efc80ede21,3,attention,F,pingvin,pingvin,passed
53,2,1,results_01.csv,2,6106fc335b80e1efc80ede21,6106fc335b80e1efc80ede21,4,attention-alt,F,vacsora,vacsora,passed
90,3,1,results_01.csv,3,5d7df958cb6cf00001399929,5d7df958cb6cf00001399929,1,attention,F,burgonya,burgonya,passed
86,3,1,results_01.csv,3,5d7df958cb6cf00001399929,5d7df958cb6cf00001399929,2,attention-alt,F,jegesmedve,jegesmedve,passed


Participants with one or more failed attention checks:
None


### 2.3. Merging with the original stimuli

Build `event_table` with one row per logged experimental keystroke (excluding practice and instructions). Trial-level choices and region times repeat across the events in a trial. The table preview is limited to 30 rows; the full dataframe remains available. `participant` is a pseudonymous submission label, but `participant_id` remains identifying. Intervals are calculated before the display sort.

Join the original stimuli on `no`, retaining unmatched events. `analysis_trials` and `plot_trials` are trial-level tables, avoiding event-level duplication. Set `FILTER_EXP_2` below to exclude `exp == 2` from these tables; it does not change attention-check reporting.

In [68]:
trial_results = ['choice_key', 'choice_side', 'choice_ab', 'chosen_image',
                 'interval_before_choice_ms'] + [f'r{i}_ms' for i in range(1, 7)]
event_table = events.merge(trials[trial_keys + trial_results], on=trial_keys,
                           how='left', validate='many_to_one')
event_table = event_table.sort_values(
    ['session', 'no', 'item', 'trial_number', 'key_number_in_trial'],
    na_position='last', kind='stable').reset_index(drop=True)
session_order = sorted(trials['session'].dropna().unique())
participant_labels = {session: f'participant_{index:02d}'
                      for index, session in enumerate(session_order, start=1)}
event_table['participant'] = event_table['session'].map(participant_labels)

# Set True to exclude events from trials where exp == 2.
FILTER_EXP_2 = False
if FILTER_EXP_2:
    event_table = event_table.loc[event_table['exp'].ne(2)].copy()

display_columns = [
    'participant', 'participant_id', 'list_number', 'source_file', 'file_session', 'no', 'cnd', 'session', 'item', 'exp', 'pst', 'mlt', 'cb',
    'stage', 'key', 'timestamp_ms', 'previous_stage', 'key_number_in_trial',
    'within_trial_ms', 'since_previous_key_ms', 'option_a', 'option_b',
    'correct', 'choice_key', 'choice_side', 'choice_ab', 'chosen_image',
    'interval_before_choice_ms', *[f'r{i}_ms' for i in range(1, 7)],
]
event_table = event_table[display_columns]
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(event_table.head(30))

# Optional private export (do not commit this file):
# event_table.to_csv('parsed_events.csv', index=False)

,participant,participant_id,list_number,source_file,file_session,no,cnd,session,item,exp,pst,mlt,cb,stage,key,timestamp_ms,previous_stage,key_number_in_trial,within_trial_ms,since_previous_key_ms,option_a,option_b,correct,choice_key,choice_side,choice_ab,chosen_image,interval_before_choice_ms,r1_ms,r2_ms,r3_ms,r4_ms,r5_ms,r6_ms
0,participant_01,69f05ebc11ace319136e99d2,1,results_01.csv,1,1,C1T1,1,1,1,0,1,n,context_key,SPACE,1793199506541,NaN,1,NaN,2464.0,1a,1b,1b,J,right,B,1b,1216.0,336.0,320.0,320.0,288.0,304.0,320.0
1,participant_01,69f05ebc11ace319136e99d2,1,results_01.csv,1,1,C1T1,1,1,1,0,1,n,r0,SPACE,1793199507277,context_key,2,736.0,736.0,1a,1b,1b,J,right,B,1b,1216.0,336.0,320.0,320.0,288.0,304.0,320.0
2,participant_01,69f05ebc11ace319136e99d2,1,results_01.csv,1,1,C1T1,1,1,1,0,1,n,r1,SPACE,1793199507613,r0,3,336.0,336.0,1a,1b,1b,J,right,B,1b,1216.0,336.0,320.0,320.0,288.0,304.0,320.0
3,participant_01,69f05ebc11ace319136e99d2,1,results_01.csv,1,1,C1T1,1,1,1,0,1,n,r2,SPACE,1793199507933,r1,4,320.0,320.0,1a,1b,1b,J,right,B,1b,1216.0,336.0,320.0,320.0,288.0,304.0,320.0
4,participant_01,69f05ebc11ace319136e99d2,1,results_01.csv,1,1,C1T1,1,1,1,0,1,n,r3,SPACE,1793199508253,r2,5,320.0,320.0,1a,1b,1b,J,right,B,1b,1216.0,336.0,320.0,320.0,288.0,304.0,320.0
5,participant_01,69f05ebc11ace319136e99d2,1,results_01.csv,1,1,C1T1,1,1,1,0,1,n,r4,SPACE,1793199508541,r3,6,288.0,288.0,1a,1b,1b,J,right,B,1b,1216.0,336.0,320.0,320.0,288.0,304.0,320.0
6,participant_01,69f05ebc11ace319136e99d2,1,results_01.csv,1,1,C1T1,1,1,1,0,1,n,r5,SPACE,1793199508845,r4,7,304.0,304.0,1a,1b,1b,J,right,B,1b,1216.0,336.0,320.0,320.0,288.0,304.0,320.0
7,participant_01,69f05ebc11ace319136e99d2,1,results_01.csv,1,1,C1T1,1,1,1,0,1,n,r6,SPACE,1793199509165,r5,8,320.0,320.0,1a,1b,1b,J,right,B,1b,1216.0,336.0,320.0,320.0,288.0,304.0,320.0
8,participant_01,69f05ebc11ace319136e99d2,1,results_01.csv,1,1,C1T1,1,1,1,0,1,n,choice_key,J,1793199510381,r6,9,1216.0,1216.0,1a,1b,1b,J,right,B,1b,1216.0,336.0,320.0,320.0,288.0,304.0,320.0
9,participant_01,69f05ebc11ace319136e99d2,1,results_01.csv,1,14,C1T2,1,2,1,0,1,y,context_key,SPACE,1793199537389,NaN,1,NaN,3136.0,2a,2b,2a,J,right,A,2a,896.0,320.0,272.0,320.0,336.0,320.0,320.0


In [69]:
event_table

,participant,participant_id,list_number,source_file,file_session,no,cnd,session,item,exp,...,choice_side,choice_ab,chosen_image,interval_before_choice_ms,r1_ms,r2_ms,r3_ms,r4_ms,r5_ms,r6_ms
0,participant_01,69f05ebc11ace319136e99d2,1,results_01.csv,1,1,C1T1,1,1,1,...,right,B,1b,1216.0,336.0,320.0,320.0,288.0,304.0,320.0
1,participant_01,69f05ebc11ace319136e99d2,1,results_01.csv,1,1,C1T1,1,1,1,...,right,B,1b,1216.0,336.0,320.0,320.0,288.0,304.0,320.0
2,participant_01,69f05ebc11ace319136e99d2,1,results_01.csv,1,1,C1T1,1,1,1,...,right,B,1b,1216.0,336.0,320.0,320.0,288.0,304.0,320.0
3,participant_01,69f05ebc11ace319136e99d2,1,results_01.csv,1,1,C1T1,1,1,1,...,right,B,1b,1216.0,336.0,320.0,320.0,288.0,304.0,320.0
4,participant_01,69f05ebc11ace319136e99d2,1,results_01.csv,1,1,C1T1,1,1,1,...,right,B,1b,1216.0,336.0,320.0,320.0,288.0,304.0,320.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14315,participant_40,undefined,4,results_04.csv,10,351,attention,40,4,0,...,right,B,reggeli,1547.0,506.0,416.0,387.0,384.0,458.0,851.0
14316,participant_40,undefined,4,results_04.csv,10,351,attention,40,4,0,...,right,B,reggeli,1547.0,506.0,416.0,387.0,384.0,458.0,851.0
14317,participant_40,undefined,4,results_04.csv,10,351,attention,40,4,0,...,right,B,reggeli,1547.0,506.0,416.0,387.0,384.0,458.0,851.0
14318,participant_40,undefined,4,results_04.csv,10,351,attention,40,4,0,...,right,B,reggeli,1547.0,506.0,416.0,387.0,384.0,458.0,851.0


In [70]:
# Join stimulus metadata without dropping unmatched logged events.
STIMULI_PATH = RESULTS_DIR / 'stimuli.csv'
stimuli = pd.read_csv(STIMULI_PATH)
stimuli['no'] = pd.to_numeric(stimuli['no'], errors='coerce').astype('Int64')
stimuli_with_no = stimuli.dropna(subset=['no']).copy()
if stimuli_with_no['no'].duplicated().any():
    raise ValueError('Stimuli must have at most one row per nonmissing no value')

# Keep result-log fields, while retaining colliding stimulus fields with a clear suffix.
overlapping_columns = (set(stimuli_with_no.columns) & set(event_table.columns)) - {'no'}
stimuli_for_join = stimuli_with_no.rename(
    columns={column: f'{column}_stimulus' for column in overlapping_columns}
)
event_rows_before_merge = len(event_table)
event_table = event_table.merge(
    stimuli_for_join, on='no', how='left', validate='many_to_one', indicator='stimuli_match'
)
if len(event_table) != event_rows_before_merge:
    raise ValueError('Stimuli merge changed the event-table row count')
matched_events = event_table['stimuli_match'].eq('both')
if not event_table.loc[matched_events, 'cnd'].eq(event_table.loc[matched_events, 'cnd_stimulus']).all():
    raise ValueError('Logged cnd does not match the stimulus cnd for one or more no values')

stimulus_conditions = stimuli_with_no[['no', 'context_condition', 'target_condition']]
analysis_trials = trials.merge(
    stimulus_conditions, on='no', how='left', validate='many_to_one'
)
if FILTER_EXP_2:
    analysis_trials = analysis_trials.loc[analysis_trials['exp'].ne(2)].copy()
plot_trials = analysis_trials.dropna(subset=['context_condition', 'target_condition']).copy()
plot_trials['participant'] = plot_trials['session'].map(participant_labels)
plot_trials['choice_rt_ms'] = plot_trials['interval_before_choice_ms']
if plot_trials.empty:
    raise ValueError('No matched stimulus trials available for condition plots')

print(f'Stimulus rows joined: {int(matched_events.sum()):,} events; unmatched events retained: {int((~matched_events).sum()):,}.')
print(f'Condition plots include {len(plot_trials):,} matched trials from {plot_trials.session.nunique()} submissions.')
display(event_table[['participant', 'participant_id', 'session', 'list_number', 'file_session', 'no', 'cnd',
                     'context_condition', 'target_condition', 'stimuli_match']].head())

Stimulus rows joined: 12,900 events; unmatched events retained: 1,420.
Condition plots include 960 matched trials from 40 submissions.


,participant,participant_id,session,list_number,file_session,no,cnd,context_condition,target_condition,stimuli_match
0,participant_01,69f05ebc11ace319136e99d2,1,1,1,1,C1T1,ctx-exhaustive,non-focused,both
1,participant_01,69f05ebc11ace319136e99d2,1,1,1,1,C1T1,ctx-exhaustive,non-focused,both
2,participant_01,69f05ebc11ace319136e99d2,1,1,1,1,C1T1,ctx-exhaustive,non-focused,both
3,participant_01,69f05ebc11ace319136e99d2,1,1,1,1,C1T1,ctx-exhaustive,non-focused,both
4,participant_01,69f05ebc11ace319136e99d2,1,1,1,1,C1T1,ctx-exhaustive,non-focused,both


## 3. Analysis

Only trials matched to context and target conditions enter the summaries and figures. The summaries are descriptive, not inferential tests. For a print-ready vector export, use e.g. `reading_figures['context_condition'].write_image('reading_times.svg')` with a working Plotly static-image export installation. Figures are 700 px wide (about 185 mm at 96 dpi); check the target journal's column size and export requirements before submission.

### 3.1. Reading times and choice intervals

#### Reading-time analysis

Calculate a mean for each submission and condition first; the displayed mean and SD/SE are then calculated across submissions. Set `ERROR_BAR` to `sd_ms` or `se_ms` to choose the error bars. Region intervals are only available when both consecutive region keys were logged.

In [71]:
ERROR_BAR = 'se_ms'  # Change to 'sd_ms' for standard-deviation bars.
condition_columns = ['cnd', 'context_condition', 'target_condition']
region_columns = [f'r{region}_ms' for region in range(1, 7)]
region_order = [f'r{region}' for region in range(1, 7)]

reading_long = plot_trials[
    ['session', 'participant', *condition_columns, *region_columns]
].melt(
    id_vars=['session', 'participant', *condition_columns],
    value_vars=region_columns,
    var_name='region', value_name='reading_time_ms'
).dropna(subset=['reading_time_ms'])
reading_long['region'] = reading_long['region'].str.replace('_ms', '', regex=False)

def summarize_across_submissions(data, group_columns, measure):
    per_submission = (
        data.groupby(['session', 'participant', *group_columns], dropna=False, observed=True)[measure]
        .mean().rename('submission_mean_ms').reset_index()
    )
    summary = (
        per_submission.groupby(group_columns, dropna=False, observed=True)
        .agg(mean_ms=('submission_mean_ms', 'mean'),
             sd_ms=('submission_mean_ms', 'std'),
             n_submissions=('session', 'nunique'))
        .reset_index()
    )
    summary['se_ms'] = summary['sd_ms'] / summary['n_submissions'].pow(0.5)
    return summary

if ERROR_BAR not in {'sd_ms', 'se_ms'}:
    raise ValueError("ERROR_BAR must be either 'sd_ms' or 'se_ms'")


# Compute submission-level means first, then summarize across submissions.
reading_summaries = {
    condition: summarize_across_submissions(reading_long, [condition, 'region'], 'reading_time_ms')
    for condition in condition_columns
}
for condition, summary in reading_summaries.items():
    print(f'Reading-time summary by {condition}:')
    display(summary)

Reading-time summary by cnd:


,cnd,region,mean_ms,sd_ms,n_submissions,se_ms
0,C1T1,r1,514.6000,146.984658,40,23.240315
1,C1T1,r2,426.3125,134.258920,40,21.228199
2,C1T1,r3,420.5625,129.976190,40,20.551040
3,C1T1,r4,466.7250,175.543258,40,27.755826
4,C1T1,r5,483.0375,185.451023,40,29.322381
...,...,...,...,...,...,...
67,C3T4,r2,503.6125,252.356903,40,39.901130
68,C3T4,r3,419.4125,142.588924,40,22.545288
69,C3T4,r4,529.7375,237.057198,40,37.482034
70,C3T4,r5,575.7375,250.774697,40,39.650961


Reading-time summary by context_condition:


,context_condition,region,mean_ms,sd_ms,n_submissions,se_ms
0,ctx-base,r1,545.715625,182.564061,40,28.865913
1,ctx-base,r2,438.421875,130.948438,40,20.704766
2,ctx-base,r3,424.990625,121.927506,40,19.278431
3,ctx-base,r4,516.562500,207.833471,40,32.861357
4,ctx-base,r5,534.221875,186.240788,40,29.447254
5,ctx-base,r6,615.018750,238.963773,40,37.783490
6,ctx-contrastive,r1,529.950000,129.750070,40,20.515287
7,ctx-contrastive,r2,442.456250,131.278649,40,20.756977
8,ctx-contrastive,r3,420.040625,115.890589,40,18.323911
9,ctx-contrastive,r4,492.784375,167.492319,40,26.482861


Reading-time summary by target_condition:


,target_condition,region,mean_ms,sd_ms,n_submissions,se_ms
0,non-focused,r1,525.283333,137.350142,40,21.716964
1,non-focused,r2,425.408333,116.609070,40,18.437513
2,non-focused,r3,435.495833,128.045573,40,20.245783
3,non-focused,r4,481.200000,175.751870,40,27.788811
4,non-focused,r5,486.858333,161.962024,40,25.608445
5,non-focused,r6,625.570833,252.184128,40,39.873812
6,pvf-base,r1,529.762500,167.277517,40,26.448898
7,pvf-base,r2,438.933333,141.332264,40,22.346593
8,pvf-base,r3,434.754167,119.817002,40,18.944731
9,pvf-base,r4,528.087500,211.425578,40,33.429319


#### Reading-time plots

Exhaustive is blue, base green, contrastive yellow, and non-focused grey. For combined `cnd` conditions, color identifies the context (C1 exhaustive, C2 base, C3 contrastive), while line style identifies the target (T1 non-focused: solid; T2 exhaustive: dashed; T3 base: dotted; T4 contrastive: dash-dot).

In [72]:
# Combined CxTy conditions use the context color and a target-specific line style.
condition_pairs = plot_trials[['cnd', 'context_condition', 'target_condition']].drop_duplicates()
if condition_pairs['cnd'].duplicated().any():
    raise ValueError('A combined condition maps to multiple context/target conditions')
missing_colors = set(condition_pairs['context_condition']) | set(condition_pairs['target_condition'])
missing_colors -= set(CONDITION_COLORS)
if missing_colors:
    raise ValueError(f'No Nord colors defined for conditions: {sorted(missing_colors)}')
missing_dashes = set(condition_pairs['target_condition']) - set(TARGET_DASHES)
if missing_dashes:
    raise ValueError(f'No line styles defined for target conditions: {sorted(missing_dashes)}')
combined_colors = {
    row.cnd: CONDITION_COLORS[row.context_condition]
    for row in condition_pairs.itertuples(index=False)
}
combined_dashes = {
    row.cnd: TARGET_DASHES[row.target_condition]
    for row in condition_pairs.itertuples(index=False)
}
color_map = {**CONDITION_COLORS, **combined_colors}

reading_figures = {}
for condition, summary in reading_summaries.items():
    figure = px.line(
        summary, x='region', y='mean_ms', color=condition,
        line_dash=condition if condition == 'cnd' else None,
        color_discrete_map=color_map, line_dash_map=combined_dashes,
        markers=True, error_y=ERROR_BAR, category_orders={'region': region_order},
        hover_data={'sd_ms': ':.1f', 'se_ms': ':.1f', 'n_submissions': True},
        labels={'region': 'Sentence region', 'mean_ms': 'Mean reading time (ms)',
                condition: condition.replace('_', ' ').title()},
        title=f'Reading times by {condition.replace("_", " ")} (±{ERROR_BAR[:2].upper()})'
    )
    figure.update_layout(width=FIGURE_WIDTH, height=FIGURE_HEIGHT,
                         margin=dict(l=65, r=20, t=55, b=55), hovermode='x unified',
                         legend=dict(orientation='h', yanchor='top', y=-0.2))
    reading_figures[condition] = figure
    figure.show()

#### Logged choice-interval analysis

This is the interval from the preceding logged key to the choice key, **not** a choice-onset RT: it includes the 500 ms fixation and may include other delays. Means and error bars use the same submission-level method as above.

In [73]:
# Summarize the logged pre-choice interval using the same submission-level method.
choice_summaries = {}
for condition in condition_columns:
    summary = summarize_across_submissions(
        plot_trials.dropna(subset=['choice_rt_ms']), [condition], 'choice_rt_ms'
    ).sort_values(condition, kind='stable')
    choice_summaries[condition] = summary
    print(f'Logged choice-interval summary by {condition}:')
    display(summary)

Logged choice-interval summary by cnd:


,cnd,mean_ms,sd_ms,n_submissions,se_ms
0,C1T1,2772.5750,1730.548281,40,273.623708
1,C1T2,1629.3500,544.395668,40,86.076513
2,C1T3,2130.2875,1308.787430,40,206.937463
3,C1T4,3022.1250,1862.964331,40,294.560524
4,C2T1,2949.1500,2797.058105,40,442.253718
5,C2T2,1787.4375,761.739526,40,120.441594
6,C2T3,4175.4125,14566.143449,40,2303.109501
7,C2T4,2744.9625,1831.396785,40,289.569257
8,C3T1,2310.8375,1330.371499,40,210.350204
9,C3T2,1994.3375,1211.305015,40,191.524139


Logged choice-interval summary by context_condition:


,context_condition,mean_ms,sd_ms,n_submissions,se_ms
0,ctx-base,2914.240625,3770.831869,40,596.220869
1,ctx-contrastive,2203.546875,947.073142,40,149.745412
2,ctx-exhaustive,2388.584375,919.159454,40,145.331870


Logged choice-interval summary by target_condition:


,target_condition,mean_ms,sd_ms,n_submissions,se_ms
0,non-focused,2677.520833,1582.066112,40,250.146616
1,pvf-base,2874.404167,5123.560511,40,810.106047
2,pvf-contrastive,2652.862500,1214.156708,40,191.975032
3,pvf-exhaustive,1803.708333,695.554745,40,109.976862


#### Logged choice-interval plots

Combined-condition bars use the context color; target conditions can be read from the T1–T4 codes described above.

In [74]:
# Use the same condition colors as the reading-time figures.
choice_interval_figures = {}
for condition, summary in choice_summaries.items():
    figure = px.bar(
        summary, x=condition, y='mean_ms', color=condition,
        color_discrete_map=color_map, error_y=ERROR_BAR,
        hover_data={'sd_ms': ':.1f', 'se_ms': ':.1f', 'n_submissions': True},
        labels={condition: condition.replace('_', ' ').title(),
                'mean_ms': 'Mean logged choice interval (ms)'},
        title=f'Logged choice intervals by {condition.replace("_", " ")} (±{ERROR_BAR[:2].upper()})'
    )
    figure.update_layout(width=FIGURE_WIDTH, height=FIGURE_HEIGHT,
                         margin=dict(l=65, r=20, t=55, b=100), showlegend=False)
    figure.update_xaxes(tickangle=-35 if condition != 'cnd' else 0)
    choice_interval_figures[condition] = figure
    figure.show()

### 3.2. Choices (behavioural)

The following counts and donut charts pool logged A/B choices across matched trials; they are not per-submission averages. A and B refer to stimulus options, not left and right screen position. No statistical model or test is fitted here.

In [75]:
# Count mapped A/B choices for each condition before plotting.
choice_data = plot_trials.dropna(subset=['choice_ab']).copy()
if choice_data.empty:
    raise ValueError('No mapped A/B choices available for behavioural analysis')
choice_counts = {}
for condition in condition_columns:
    choice_counts[condition] = (
        choice_data.groupby([condition, 'choice_ab'], observed=True)
        .size().unstack(fill_value=0).reindex(columns=['A', 'B'], fill_value=0)
    )
    print(f'Pooled choice counts by {condition}:')
    display(choice_counts[condition])

Pooled choice counts by cnd:


choice_ab,A,B
cnd,,
C1T1,40,40
C1T2,80,0
C1T3,78,2
C1T4,59,21
C2T1,40,40
C2T2,80,0
C2T3,80,0
C2T4,49,31
C3T1,48,32


Pooled choice counts by context_condition:


choice_ab,A,B
context_condition,,
ctx-base,249,71
ctx-contrastive,252,68
ctx-exhaustive,257,63


Pooled choice counts by target_condition:


choice_ab,A,B
target_condition,,
non-focused,128,112
pvf-base,233,7
pvf-contrastive,162,78
pvf-exhaustive,235,5


#### Choice plots

Each donut shows pooled choice proportions within one condition. Slice colors distinguish A from B and do not encode exhaustive/base/contrastive conditions.

In [76]:
# Plot pooled A/B counts; the slices use choice colors rather than condition colors.
def choice_donut_figure(data, condition):
    levels = sorted(data[condition].dropna().unique(), key=str)
    columns = min(3, len(levels))
    rows = math.ceil(len(levels) / columns)
    specs = [
        [{'type': 'domain'} if column < len(levels) - row * columns else None
         for column in range(columns)]
        for row in range(rows)
    ]
    figure = make_subplots(
        rows=rows, cols=columns, specs=specs,
        subplot_titles=[str(level) for level in levels]
    )
    for index, level in enumerate(levels):
        row, column = divmod(index, columns)
        counts = (
            data.loc[data[condition].eq(level), 'choice_ab']
            .value_counts().reindex(['A', 'B'], fill_value=0)
        )
        figure.add_trace(
            go.Pie(
                labels=counts.index, values=counts.values, hole=0.5, sort=False,
                textinfo='label+percent', showlegend=index == 0,
                marker={'colors': CHOICE_COLORS},
                hovertemplate='<b>Choice %{label}</b><br>Trials: %{value}<br>Share: %{percent}<extra></extra>'
            ), row=row + 1, col=column + 1
        )
    figure.update_layout(
        title=f'A/B choices by {condition.replace("_", " ")}',
        height=max(300, rows * 200 + 100), width=FIGURE_WIDTH,
        legend_title_text='Choice', margin={'l': 20, 'r': 20, 't': 75, 'b': 20}
    )
    return figure

choice_donut_figures = {}
for condition in condition_columns:
    choice_donut_figures[condition] = choice_donut_figure(choice_data, condition)
    choice_donut_figures[condition].show()

## 4. Focused-target timing comparisons

The following analyses include only the exhaustive (T2), base (T3), and contrastive (T4) target conditions; non-focused targets (T1) are excluded. They use the existing `plot_trials`, including its `FILTER_EXP_2` setting. Repeated participants and attention-check failures are not automatically excluded.

For each measure, compare targets pooled across contexts and separately within each context. Descriptive means average trials within a submission and then average submissions; pooled means may weight contexts unequally within a submission. Error bars use the earlier `ERROR_BAR` setting. Missing component times make the corresponding measure missing; no partial average is substituted. Reading times are logged key-to-key intervals, not display-onset times.

In [77]:
# Select focused targets and calculate all three measures at the trial level.
focused_targets = ['pvf-exhaustive', 'pvf-base', 'pvf-contrastive']
focused_contexts = ['ctx-exhaustive', 'ctx-base', 'ctx-contrastive']
focused_trials = plot_trials.loc[plot_trials['target_condition'].isin(focused_targets)].copy()
if focused_trials.empty:
    raise ValueError('No focused-target trials are available for the reading-time comparisons')
unexpected_contexts = set(focused_trials['context_condition']) - set(focused_contexts)
if unexpected_contexts:
    raise ValueError(f'Unexpected contexts in focused trials: {sorted(unexpected_contexts)}')

focused_trials['object_r4_ms'] = focused_trials['r4_ms']
focused_trials['modifier_object_ms'] = pd.NA
is_exhaustive = focused_trials['target_condition'].eq('pvf-exhaustive')
is_base = focused_trials['target_condition'].eq('pvf-base')
is_contrastive = focused_trials['target_condition'].eq('pvf-contrastive')
focused_trials.loc[is_exhaustive, 'modifier_object_ms'] = (
    focused_trials.loc[is_exhaustive, 'r3_ms'] + focused_trials.loc[is_exhaustive, 'r4_ms']
) / 2
focused_trials.loc[is_base, 'modifier_object_ms'] = focused_trials.loc[is_base, 'r4_ms']
focused_trials.loc[is_contrastive, 'modifier_object_ms'] = (
    focused_trials.loc[is_contrastive, 'r2_ms'] + focused_trials.loc[is_contrastive, 'r3_ms']
    + focused_trials.loc[is_contrastive, 'r4_ms']
) / 3
focused_trials['modifier_object_ms'] = pd.to_numeric(focused_trials['modifier_object_ms'])
focused_trials['verb_r5_ms'] = focused_trials['r5_ms']
print(f'Focused trials: {len(focused_trials):,} from {focused_trials.session.nunique()} submissions')
print('Missing measures:')
display(focused_trials[['object_r4_ms', 'modifier_object_ms', 'verb_r5_ms', 'interval_before_choice_ms']].isna().sum())

# Reuse the earlier per-submission summary; plot pooled and context-specific results.
focused_rt_summaries = {}
focused_rt_figures = {}
def compare_focused_reading(metric, title, y_label='Mean reading time (ms)'):
    available = focused_trials.dropna(subset=[metric])
    if available.empty:
        raise ValueError(f'No available reading times for {metric}')
    pooled = summarize_across_submissions(available, ['target_condition'], metric)
    by_context = summarize_across_submissions(
        available, ['context_condition', 'target_condition'], metric
    )
    focused_rt_summaries[metric] = {'pooled': pooled, 'by_context': by_context}
    print(f'{title}: pooled across contexts')
    display(pooled)
    print(f'{title}: by context and target')
    display(by_context)

    figures = {}
    for name, summary in [('pooled', pooled), ('by_context', by_context)]:
        figure = px.bar(
            summary, x='target_condition', y='mean_ms', color='target_condition',
            facet_col='context_condition' if name == 'by_context' else None,
            color_discrete_map=CONDITION_COLORS, error_y=ERROR_BAR,
            category_orders={'target_condition': focused_targets, 'context_condition': focused_contexts},
            hover_data={'sd_ms': ':.1f', 'se_ms': ':.1f', 'n_submissions': True},
            labels={'target_condition': 'Target condition', 'mean_ms': y_label},
            title=f'{title}: {"by context" if name == "by_context" else "pooled contexts"} (±{ERROR_BAR[:2].upper()})'
        )
        figure.update_layout(width=FIGURE_WIDTH, height=FIGURE_HEIGHT, showlegend=False,
                             margin=dict(l=65, r=20, t=70, b=100))
        figure.update_xaxes(tickangle=-35)
        figure.for_each_annotation(lambda annotation: annotation.update(
            text=annotation.text.replace('context_condition=', '')
        ))
        figures[name] = figure
        figure.show()
    focused_rt_figures[metric] = figures

Focused trials: 720 from 40 submissions
Missing measures:


object_r4_ms                 0
modifier_object_ms           0
verb_r5_ms                   0
interval_before_choice_ms    0
dtype: int64

### 4.1. Objects

Compare the object region (`r4_ms`) across the three focused target conditions.

In [78]:
# Summarize and plot object-region reading times.
compare_focused_reading('object_r4_ms', 'Object (r4)')

Object (r4): pooled across contexts


,target_condition,mean_ms,sd_ms,n_submissions,se_ms
0,pvf-base,528.087500,211.425578,40,33.429319
1,pvf-contrastive,534.458333,217.017920,40,34.313546
2,pvf-exhaustive,487.370833,194.655201,40,30.777690


Object (r4): by context and target


,context_condition,target_condition,mean_ms,sd_ms,n_submissions,se_ms
0,ctx-base,pvf-base,494.8625,165.369826,40,26.147265
1,ctx-base,pvf-contrastive,549.4625,266.423039,40,42.125181
2,ctx-base,pvf-exhaustive,516.3125,269.670114,40,42.638589
3,ctx-contrastive,pvf-base,515.5125,210.116192,40,33.222287
4,ctx-contrastive,pvf-contrastive,529.7375,237.057198,40,37.482034
5,ctx-contrastive,pvf-exhaustive,454.6250,169.873813,40,26.859408
6,ctx-exhaustive,pvf-base,573.8875,368.347026,40,58.240779
7,ctx-exhaustive,pvf-contrastive,524.1750,236.122403,40,37.334230
8,ctx-exhaustive,pvf-exhaustive,491.1750,194.562611,40,30.763050


### 4.2. Modifiers and objects

The per-trial measure is `(r3 + r4) / 2` for exhaustive targets (`csak`), `r4` for base targets, and `(r2 + r3 + r4) / 3` for contrastive targets (`többek között`). The number of regions differs by target condition, so these are averages per included region, not total phrase times.

In [79]:
# Summarize and plot the condition-specific modifier-plus-object averages.
compare_focused_reading('modifier_object_ms', 'Modifier and object')

Modifier and object: pooled across contexts


,target_condition,mean_ms,sd_ms,n_submissions,se_ms
0,pvf-base,528.087500,211.425578,40,33.429319
1,pvf-contrastive,482.286111,162.780539,40,25.737863
2,pvf-exhaustive,442.404167,152.559681,40,24.121804


Modifier and object: by context and target


,context_condition,target_condition,mean_ms,sd_ms,n_submissions,se_ms
0,ctx-base,pvf-base,494.862500,165.369826,40,26.147265
1,ctx-base,pvf-contrastive,493.320833,188.947604,40,29.875239
2,ctx-base,pvf-exhaustive,456.712500,182.901723,40,28.919302
3,ctx-contrastive,pvf-base,515.512500,210.116192,40,33.222287
4,ctx-contrastive,pvf-contrastive,484.254167,168.157685,40,26.588065
5,ctx-contrastive,pvf-exhaustive,422.143750,134.433079,40,21.255736
6,ctx-exhaustive,pvf-base,573.887500,368.347026,40,58.240779
7,ctx-exhaustive,pvf-contrastive,469.283333,185.346850,40,29.305910
8,ctx-exhaustive,pvf-exhaustive,448.356250,168.697631,40,26.673437


### 4.3. Verbs

Compare the verb region (`r5_ms`) across the three focused target conditions.

In [80]:
# Summarize and plot verb-region reading times.
compare_focused_reading('verb_r5_ms', 'Verb (r5)')

Verb (r5): pooled across contexts


,target_condition,mean_ms,sd_ms,n_submissions,se_ms
0,pvf-base,550.591667,195.557301,40,30.920324
1,pvf-contrastive,561.050000,215.267640,40,34.036803
2,pvf-exhaustive,525.650000,207.186167,40,32.759009


Verb (r5): by context and target


,context_condition,target_condition,mean_ms,sd_ms,n_submissions,se_ms
0,ctx-base,pvf-base,545.1375,188.418757,40,29.791621
1,ctx-base,pvf-contrastive,534.9625,213.748666,40,33.796632
2,ctx-base,pvf-exhaustive,554.0750,277.111163,40,43.815122
3,ctx-contrastive,pvf-base,540.6875,200.278258,40,31.666773
4,ctx-contrastive,pvf-contrastive,575.7375,250.774697,40,39.650961
5,ctx-contrastive,pvf-exhaustive,507.4000,216.141964,40,34.175045
6,ctx-exhaustive,pvf-base,565.9500,272.983817,40,43.162531
7,ctx-exhaustive,pvf-contrastive,572.4500,256.067344,40,40.487802
8,ctx-exhaustive,pvf-exhaustive,515.4750,191.736797,40,30.316250


### 4.4. Logged choice intervals

Compare the interval from the preceding logged key to the choice key, pooled across contexts and within each context. This interval includes the 500 ms fixation and may include other delays; it is **not** a true choice-onset reaction time because choice-display onset was not logged. Trials without a logged choice have a missing interval.

In [81]:
# Summarize and plot the logged interval preceding each choice.
compare_focused_reading('interval_before_choice_ms', 'Logged choice interval',
                        y_label='Mean logged choice interval (ms)')

Logged choice interval: pooled across contexts


,target_condition,mean_ms,sd_ms,n_submissions,se_ms
0,pvf-base,2874.404167,5123.560511,40,810.106047
1,pvf-contrastive,2652.862500,1214.156708,40,191.975032
2,pvf-exhaustive,1803.708333,695.554745,40,109.976862


Logged choice interval: by context and target


,context_condition,target_condition,mean_ms,sd_ms,n_submissions,se_ms
0,ctx-base,pvf-base,4175.4125,14566.143449,40,2303.109501
1,ctx-base,pvf-contrastive,2744.9625,1831.396785,40,289.569257
2,ctx-base,pvf-exhaustive,1787.4375,761.739526,40,120.441594
3,ctx-contrastive,pvf-base,2317.5125,1303.698907,40,206.132896
4,ctx-contrastive,pvf-contrastive,2191.5000,1015.607965,40,160.581719
5,ctx-contrastive,pvf-exhaustive,1994.3375,1211.305015,40,191.524139
6,ctx-exhaustive,pvf-base,2130.2875,1308.787430,40,206.937463
7,ctx-exhaustive,pvf-contrastive,3022.1250,1862.964331,40,294.560524
8,ctx-exhaustive,pvf-exhaustive,1629.3500,544.395668,40,86.076513


### 4.5. Export for statistical analysis in R

Run the next cell to write `focused_rt_data.csv` beside `stimuli.csv` (overwriting an existing file of that name). It contains one row per focused trial, a submission-level pseudonym (`session`), item and condition codes, and the timing measures; it omits direct participant identifiers. Keep the file private.

Then run `Rscript new/focused_rt_analysis.R` from the repository root, or `Rscript focused_rt_analysis.R` from `new/`. The script requires the R packages `lme4`, `lmerTest`, `car`, and `emmeans`, and writes `focused_rt_analysis_results.txt` beside the input CSV, overwriting an existing report. Optional first and second arguments specify the input and report paths.

For each measure, including the logged choice interval, the script fits both a pooled target model and a target × context model, with random intercepts for submission and item. It models `log1p` of each trial-level measure, reports type-III tests with sum contrasts and Tukey-adjusted pairwise comparisons, and flags singular fits. For modifier-object times, the arithmetic average is formed **before** the log transformation. Pairwise adjustments are within each comparison family, not across measures or models. Review model diagnostics and exclusions before drawing conclusions.

In [82]:
# Export trial-level pseudonymous measures for the accompanying R script.
focused_rt_export_columns = [
    'session', 'list_number', 'file_session', 'trial_number', 'no', 'item', 'exp',
    'cnd', 'context_condition', 'target_condition', 'r2_ms', 'r3_ms', 'r4_ms', 'r5_ms',
    'object_r4_ms', 'modifier_object_ms', 'verb_r5_ms', 'interval_before_choice_ms'
]
focused_rt_path = RESULTS_DIR / 'focused_rt_data.csv'
focused_trials[focused_rt_export_columns].to_csv(focused_rt_path, index=False)
print(f'Wrote {len(focused_trials):,} focused trials to {focused_rt_path}')

Wrote 720 focused trials to focused_rt_data.csv
